# Returns on Russian corporate floating-rate bonds

Reproducible analysis in one notebook: descriptive statistics, the Fama-MacBeth
cross-section, and the Fama-French portfolio sorts.

The notebook reads two Excel files. The bond panel
`python_ff1992_final.xlsx` (sheet `FF 1992`) feeds Part A and Part B.
The portfolio file `portfolios_FINAL_2x3.xlsx` feeds Part C. 

## Setup and data

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Input files (size variable renamed to issue amount)
PANEL_FILE  = "python_ff1992_final_7june.xlsx"
PANEL_SHEET = "FF 1992"
PORT_FILE   = "portfolios_FINAL_2x3_7june.xlsx"

# Estimation parameters
MIN_OBS_PER_MONTH = 30   # months kept in each Fama-MacBeth cross-section
MIN_OBS_FOR_BETA  = 12   # minimum months to estimate a bond beta
NW_LAG_FM         = 4    # Newey-West lag, Fama-MacBeth
NW_LAG_PORT       = 3    # Newey-West lag, portfolio regressions

RENAME = {
    "Level of rating":                          "rating_level",
    "Ln(outstanding)":                          "ln_issue_amount",
    "Relative bid-ask spread":                  "rel_bid_ask",
    "Time to maturity in year":                 "ttm_year_raw",
    "Call_dummy":                               "call_dummy",
    "Put_dummy":                                "put_dummy",
    "CBI RU Total Market Investable G-Spread":  "GS_market",
}

def sig(t):
    a = abs(t)
    return "***" if a > 2.58 else "**" if a > 1.96 else "*" if a > 1.64 else ""

panel = pd.read_excel(PANEL_FILE, sheet_name=PANEL_SHEET)
panel["Date"] = pd.to_datetime(panel["Date"])
panel = panel.sort_values(["ISIN", "Date"]).reset_index(drop=True)
panel = panel.rename(columns={k: v for k, v in RENAME.items() if k in panel.columns})
panel["ln_amihud_DM"] = panel["Amihud_DM"]
panel["DM_next"] = panel.groupby("ISIN")["DM_NEW"].shift(-1)

print(f"Panel loaded: {len(panel):,} obs, {panel['ISIN'].nunique()} ISIN, "
      f"{panel['Date'].nunique()} months "
      f"({panel['Date'].min():%Y-%m} to {panel['Date'].max():%Y-%m})")

Panel loaded: 6,500 obs, 327 ISIN, 52 months (2022-01 to 2026-04)


## Part A. Descriptive statistics, correlations, multicollinearity

Reproduces Tables 2.2 to 2.5.

In [5]:
desc_vars = ["DM_NEW", "DM_next", "rating_level", "ln_issue_amount",
             "rel_bid_ask", "ln_amihud_DM", "ttm_year_raw", "call_dummy", "put_dummy"]
regressors = ["rating_level", "ln_issue_amount", "rel_bid_ask",
              "ln_amihud_DM", "ttm_year_raw", "call_dummy", "put_dummy"]

def describe(s):
    s = s.dropna()
    q = s.quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    return pd.Series({"N": int(s.size), "mean": s.mean(), "std": s.std(),
                      "p5": q[0.05], "p25": q[0.25], "median": q[0.50],
                      "p75": q[0.75], "p95": q[0.95], "max": s.max(),
                      "skew": s.skew(), "kurt": s.kurt()})

table_2_2 = pd.DataFrame({v: describe(panel[v]) for v in desc_vars}).T
table_2_2["N"] = table_2_2["N"].astype(int)

corr_2_3 = (panel[regressors + ["DM_next"]].corr()["DM_next"]
            .drop("DM_next").sort_values(ascending=False))

corr_2_4 = panel[regressors].corr()

def vif(cols):
    d = panel[cols].dropna()
    X = sm.add_constant(d.values)
    return pd.Series([variance_inflation_factor(X, i + 1) for i in range(len(cols))], index=cols)

final_cols = ["rating_level", "ln_issue_amount", "ln_amihud_DM",
              "ttm_year_raw", "call_dummy", "put_dummy"]
table_2_5 = pd.concat([vif(final_cols).rename("VIF_final"),
                       vif(final_cols + ["rel_bid_ask"]).rename("VIF_extended")], axis=1)

print("Table 2.2  Descriptive statistics\n")
print(table_2_2.round(2).to_string())
print("\nTable 2.3  Correlation with the dependent variable DM_next\n")
print(corr_2_3.round(3).to_string())
print("\nTable 2.4  Correlation matrix of the regressors\n")
print(corr_2_4.round(3).to_string())
print("\nTable 2.5  Variance inflation factors\n")
print(table_2_5.round(2).to_string())

Table 2.2  Descriptive statistics

                    N     mean      std      p5      p25   median      p75       p95       max    skew     kurt
DM_NEW           6500 403.0600 438.6300 79.0000 148.3600 240.5400 500.1400 1202.4100 5800.7200  3.7900  26.4900
DM_next          6173 409.1600 446.4100 79.3700 149.1800 242.9900 514.0600 1219.8900 5800.7200  3.7500  25.7200
rating_level     6309   4.3500   4.2400  1.0000   1.0000   3.0000   7.0000   14.0000   16.0000  1.2000   0.3200
ln_issue_amount  6500  22.3800   1.8700 18.8300  21.2400  22.8000  23.7200   24.6400   25.3400 -0.7200  -0.3900
rel_bid_ask      6196   0.0000   0.0100  0.0000   0.0000   0.0000   0.0000    0.0200    0.3800 10.2600 160.6100
ln_amihud_DM     5512   3.2100   3.3300 -2.4300   1.1300   3.3400   5.4100    8.5000   13.4000 -0.2700   0.5600
ttm_year_raw     6500   3.0900   2.3800  0.7800   1.6700   2.4900   3.6800    8.4600   30.3500  3.2300  23.5300
call_dummy       6500   0.0800   0.2700  0.0000   0.0000   0.0000   0

## Part B. Fama-MacBeth cross-section

The dependent variable is the next-month discount margin `DM_next`. Each month a
cross-sectional regression is run on bond characteristics, keeping months with at
least 30 bonds. The reported t-statistics are Newey-West corrected on the time
series of the monthly coefficients.

The two systematic loadings are estimated in a first step as full-sample slopes on
first differences, one value per bond: `beta_ext` on the change in the external CBI
G-Spread market factor, and `beta_int` on the change in the internal leave-one-out
sample mean discount margin. Reproduces Models 1 to 4.

In [7]:
fm = panel.copy()

# internal market factor: leave-one-out cross-sectional mean of DM_NEW
grp = fm.dropna(subset=["DM_NEW"]).groupby("Date")["DM_NEW"]
fm["loo_mean"] = (fm["Date"].map(grp.sum()) - fm["DM_NEW"]) / (fm["Date"].map(grp.count()) - 1)
fm.loc[fm["DM_NEW"].isna(), "loo_mean"] = np.nan

# first differences for the beta step
fm["dDM"]  = fm.groupby("ISIN")["DM_NEW"].diff()
fm["dGS"]  = fm.groupby("ISIN")["GS_market"].diff()
fm["dLOO"] = fm.groupby("ISIN")["loo_mean"].diff()

def bond_beta(ychg, xchg):
    """Full-sample slope of one differenced series on another, one value per bond."""
    out = {}
    for isin, g in fm.groupby("ISIN"):
        s = g[[ychg, xchg]].dropna()
        if len(s) >= MIN_OBS_FOR_BETA:
            out[isin] = sm.OLS(s[ychg], sm.add_constant(s[xchg])).fit().params[xchg]
    return pd.Series(out)

fm["beta_ext"] = fm["ISIN"].map(bond_beta("dDM", "dGS"))
fm["beta_int"] = fm["ISIN"].map(bond_beta("dDM", "dLOO"))

def fama_macbeth(regressors, dep="DM_next", lag=NW_LAG_FM):
    coefs = []
    for _, g in fm.groupby("Date"):
        s = g[[dep] + regressors].dropna()
        if len(s) >= MIN_OBS_PER_MONTH:
            coefs.append(sm.OLS(s[dep], sm.add_constant(s[regressors])).fit().params)
    coefs = pd.DataFrame(coefs)
    rows = []
    for name in coefs.columns:
        b = coefs[name].dropna().values
        nw = sm.OLS(b, np.ones(len(b))).fit(cov_type="HAC", cov_kwds={"maxlags": lag})
        rows.append((name, b.mean(), nw.tvalues[0], sig(nw.tvalues[0])))
    table = pd.DataFrame(rows, columns=["Variable", "Mean coef", "t_NW", "sig"]).set_index("Variable")
    return table, len(coefs)

chars = ["rating_level", "ln_issue_amount", "ln_amihud_DM",
         "ttm_year_raw", "call_dummy", "put_dummy"]
models = {"Model 1 (characteristics)": chars,
          "Model 2 (+ external beta)": chars + ["beta_ext"],
          "Model 3 (+ internal beta)": chars + ["beta_int"],
          "Model 4 (+ both betas)":    chars + ["beta_ext", "beta_int"]}

for title, regs in models.items():
    table, T = fama_macbeth(regs)
    print(f"{title}   (months T = {T}, Newey-West lag {NW_LAG_FM})\n")
    print(table.round(2).to_string())
    print()

Model 1 (characteristics)   (months T = 29, Newey-West lag 4)

                 Mean coef    t_NW  sig
Variable                               
const            -187.8300 -1.0000     
rating_level       59.4900  5.3200  ***
ln_issue_amount    13.6300  1.6400    *
ln_amihud_DM        5.9100  2.0500   **
ttm_year_raw      -14.4000 -2.0200   **
call_dummy        116.4100  3.2500  ***
put_dummy         -22.7700 -1.3400     

Model 2 (+ external beta)   (months T = 29, Newey-West lag 4)

                 Mean coef    t_NW  sig
Variable                               
const             -95.7500 -0.6100     
rating_level       46.2300  5.9600  ***
ln_issue_amount     7.6900  1.2100     
ln_amihud_DM        4.4600  1.9700   **
ttm_year_raw       -0.6400 -0.1700     
call_dummy         78.1900  4.1100  ***
put_dummy         -53.2600 -3.4900  ***
beta_ext          415.7700  3.8800  ***

Model 3 (+ internal beta)   (months T = 29, Newey-West lag 4)

                 Mean coef    t_NW  sig
Variable 

## Part C. Fama-French portfolio sorts and GRS test

Two-by-three sorts on credit and a second characteristic. Each portfolio discount
margin is regressed on the factors in first differences with Newey-West errors, and
the joint pricing-error restriction is tested with the GRS statistic. One main and
two robustness specifications.

In [9]:
def load_spec(sheet):
    s = pd.read_excel(PORT_FILE, sheet_name=sheet)
    s["Date"] = pd.to_datetime(s["Date"])
    s = s.set_index("Date").sort_index()
    ports   = [c for c in s.columns if c.startswith("DM_")]
    factors = [c for c in s.columns if c not in ports]
    return s, ports, factors

def portfolio_regressions(data, ports, factors, lag=NW_LAG_PORT):
    dd = data[ports + factors].diff().dropna()
    rows = []
    for pc in ports:
        X  = sm.add_constant(dd[factors].values)
        r  = sm.OLS(dd[pc].values, X).fit(cov_type="HAC", cov_kwds={"maxlags": lag})
        ro = sm.OLS(dd[pc].values, X).fit()
        row = {"Portfolio": pc, "N": len(dd), "R2": r.rsquared,
               "alpha": r.params[0], "t_alpha": r.tvalues[0], "sig": sig(r.tvalues[0])}
        for i, f in enumerate(factors, start=1):
            row[f"beta_{f}"] = r.params[i]
            row[f"t_{f}"]    = r.tvalues[i]
            row[f"sig_{f}"]  = sig(r.tvalues[i])
        row["DW"] = durbin_watson(ro.resid)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Portfolio")

def grs_test(data, ports, factors):
    dd = data[ports + factors].diff().dropna()
    alpha, resid = [], []
    for pc in ports:
        ro = sm.OLS(dd[pc].values, sm.add_constant(dd[factors].values)).fit()
        alpha.append(ro.params[0]); resid.append(ro.resid)
    T, N, K = len(dd), len(ports), len(factors)
    a = np.array(alpha)
    sigma = (np.array(resid) @ np.array(resid).T) / T
    f = dd[factors].values
    fmean, fcov = f.mean(0), np.cov(f.T, ddof=1).reshape(K, K)
    F = ((T - N - K) / N) * (a @ np.linalg.inv(sigma) @ a) / (1 + fmean @ np.linalg.inv(fcov) @ fmean)
    return float(F), float(1 - stats.f.cdf(F, N, T - N - K)), T

specs = [("Main (Credit + Maturity)",              "Main_Credit_Maturity"),
         ("Robustness 1 (Credit + Issue amount)",   "Rob1_Credit_IssueAmount"),
         ("Robustness 2 (Credit + Amihud)",         "Rob2_Credit_Amihud")]

grs_rows = []
for title, sheet in specs:
    data, ports, factors = load_spec(sheet)
    print(f"{title}   (Newey-West lag {NW_LAG_PORT})\n")
    print(portfolio_regressions(data, ports, factors).round(3).to_string())
    print()
    F, p, T = grs_test(data, ports, factors)
    grs_rows.append((title, T, len(ports), len(factors), round(F, 4), round(p, 4),
                     "not rejected" if p > 0.05 else "rejected"))

print("GRS test summary\n")
print(pd.DataFrame(grs_rows, columns=["Specification", "T", "N", "K",
                                      "GRS F", "GRS p", "H0 at 5%"]).to_string(index=False))

factor_series = pd.read_excel(PORT_FILE, sheet_name="Factor_series").drop(columns="Date")
print("\nFactor correlation matrix\n")
print(factor_series.corr().round(3).to_string())

Main (Credit + Maturity)   (Newey-West lag 3)

            N     R2   alpha  t_alpha sig  beta_Credit  t_Credit sig_Credit  beta_Maturity  t_Maturity sig_Maturity     DW
Portfolio                                                                                                                 
DM_HC_LM   22 0.2270 10.4770   1.1200           0.0080    0.1250                   -0.8190     -3.2480          *** 2.1480
DM_HC_MM   22 0.3380  4.5220   0.3900           0.2150    2.3070         **        -0.6680     -3.3020          *** 1.9240
DM_HC_HM   22 0.3130  1.9470   0.2410           0.1390    2.2760         **        -0.2720     -1.7190            * 1.2410
DM_LC_LM   22 0.9010  3.1590   0.3210           1.3080   10.8770        ***        -1.8470     -5.1720          *** 1.6340
DM_LC_MM   22 0.6630  8.2080   0.6380           0.8930    6.5060        ***         0.1610      0.4950              2.3330
DM_LC_HM   22 0.0680 12.8380   0.6900           0.1160    0.5000                   -0.6310  